# 05 — Batch Enrich Papers with OpenAlex

This notebook shows a realistic workflow:
1. Load the example paper dataset (with Dimensions export data)
2. For each paper DOI, query OpenAlex to get up-to-date metadata
3. Merge the enriched fields back into the DataFrame
4. Export the enriched dataset to a new CSV

This is the kind of pipeline used in systematic reviews to enrich citation databases.

In [ ]:
import requests
import pandas as pd
import time

In [ ]:
MAILTO = 'your.email@example.com'

## 1. Load example papers

In [ ]:
df = pd.read_csv('../data/example_papers.csv')
print(f'Loaded {len(df)} papers')
df[['DOI', 'Title', 'Times cited', 'Inclusion decision']].head()

## 2. Helper: fetch one paper from OpenAlex

In [ ]:
def fetch_openalex_work(doi: str, mailto: str = '') -> dict | None:
    """Fetch an OpenAlex work by DOI. Returns None on failure."""
    url = f'https://api.openalex.org/works/https://doi.org/{doi}'
    try:
        response = requests.get(url, params={'mailto': mailto}, timeout=10)
        if response.status_code == 200:
            return response.json()
        return None
    except Exception as e:
        print(f'  Error fetching {doi}: {e}')
        return None


def extract_fields(work: dict | None) -> dict:
    """Extract the fields we want to add from the OpenAlex work object."""
    if work is None:
        return {
            'oa_cited_by_count': None,
            'oa_is_oa': None,
            'oa_oa_status': None,
            'oa_concepts': None,
            'oa_first_author_id': None,
            'oa_first_author_orcid': None,
        }

    authorships = work.get('authorships', [])
    first_author_id = None
    first_author_orcid = None
    if authorships and authorships[0].get('author'):
        first_author_id = authorships[0]['author'].get('id', '').split('/')[-1]
        first_author_orcid = authorships[0]['author'].get('orcid')

    top_concepts = ', '.join(
        c['display_name']
        for c in sorted(work.get('concepts', []), key=lambda x: -x.get('score', 0))[:3]
    )

    return {
        'oa_cited_by_count': work.get('cited_by_count'),
        'oa_is_oa': work.get('open_access', {}).get('is_oa'),
        'oa_oa_status': work.get('open_access', {}).get('oa_status'),
        'oa_concepts': top_concepts,
        'oa_first_author_id': first_author_id,
        'oa_first_author_orcid': first_author_orcid,
    }

## 3. Run the batch enrichment loop

In [ ]:
enriched_rows = []

for i, row in df.iterrows():
    doi = row['DOI']
    print(f'[{i+1}/{len(df)}] Fetching: {doi}')

    work = fetch_openalex_work(doi, mailto=MAILTO)
    fields = extract_fields(work)
    enriched_rows.append(fields)

    time.sleep(0.2)  # polite rate limiting

print('\nDone!')

## 4. Merge and inspect enriched DataFrame

In [ ]:
df_enriched = pd.concat([df, pd.DataFrame(enriched_rows)], axis=1)

# Compare Dimensions citation count vs OpenAlex
compare = df_enriched[['Title', 'Times cited', 'oa_cited_by_count', 'oa_oa_status', 'oa_concepts']].copy()
compare['Title'] = compare['Title'].str[:60]
compare

In [ ]:
# How often do citation counts agree?
df_enriched['cite_diff'] = df_enriched['oa_cited_by_count'] - df_enriched['Times cited']
print('Citation count difference (OpenAlex - Dimensions):')
print(df_enriched['cite_diff'].describe())

## 5. Export enriched dataset

In [ ]:
output_path = '../data/example_papers_enriched.csv'
df_enriched.drop(columns=['cite_diff']).to_csv(output_path, index=False)
print(f'Saved enriched dataset to {output_path}')
print(f'Shape: {df_enriched.shape}')